In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
#plotting packages
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [3]:
import sys
print(sys.path) 
sys.path.append("/cellarold/users/mpagadal/Programs/anaconda3/lib/python3.7/site-packages")

['/mnt/beegfs/users/mpagadal/projects/TestosteroneGWAS/cell-type-specificity/twas', '/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python37.zip', '/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7', '/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/lib-dynload', '', '/cellar/users/mpagadal/.local/lib/python3.7/site-packages', '/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/site-packages', '/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/site-packages/IPython/extensions', '/mnt/beegfs/users/mpagadal/.ipython']


In [4]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from matplotlib_venn import venn3

In [5]:
def rsid_map(df,rsid):
    '''
    Function to map to rsid using hg19 rsid map, taking into account allele
    '''
    
    df["snp_noallele"]=df["ID"].str.rsplit(":",2).str[0]
    rsid_filt=rsid[rsid["variant"].isin(df["snp_noallele"].tolist())]
    
    rsid_filt["snp1"]=rsid_filt[0].astype(str)+":"+rsid_filt[1].astype(str)+":"+rsid_filt[3]+":"+rsid_filt[4]
    rsid_filt["snp2"]=rsid_filt[0].astype(str)+":"+rsid_filt[1].astype(str)+":"+rsid_filt[4]+":"+rsid_filt[3]
    
    mp_snp1=dict(zip(rsid_filt["snp1"],rsid_filt[5]))
    mp_snp2=dict(zip(rsid_filt["snp2"],rsid_filt[5]))
    
    mp_rsid=Merge(mp_snp1,mp_snp2)
    
    df["rsid"]=df["ID"].map(mp_rsid)
    df=df[~df["rsid"].isnull()]
    df["ID"]=df["rsid"]
    del df["rsid"]
    return(df)

## Make TWAS input

In [6]:
# eur=pd.read_csv("/cellar/users/mpagadal/projects/TestosteroneGWAS/heritability/ldsc/sumstats/eur.testosterone.sumstats.gz",delimiter="\t")
# print(eur.shape)
# eur.to_csv("/cellar/users/mpagadal/Programs/fusion_twas-master/eur.sumstats.tsv",index=None,sep="\t")

In [7]:
# ukbb=pd.read_csv("/cellar/users/mpagadal/projects/TestosteroneGWAS/heritability/ldsc/sumstats/ukbb.testosterone.w_hm3.sumstats.gz",delimiter="\t")
# ukbb=ukbb.dropna()
# print(ukbb.shape)
# ukbb.to_csv("/cellar/users/mpagadal/Programs/fusion_twas-master/ukbb.sumstats.tsv",index=None,sep="\t")

## Get TWAS input

In [14]:
# twas=pd.read_csv("/cellar/users/mpagadal/Programs/fusion_twas-master/eur.sumstats.tsv",delimiter="\t")

# #get ldref input
# files=[x for x in os.listdir("/cellar/users/mpagadal/Data2/ldsc/1000G_plink/EUR") if x.endswith("bim")]
# files=[x for x in files if "CM" not in x]

# bim=pd.read_csv("/cellar/users/mpagadal/Data2/ldsc/1000G_plink/EUR/"+files[0],header=None,delimiter="\t")

# compiled_bim=pd.DataFrame()

# for x in files:
#     print(x)
#     bim=pd.read_csv("/cellar/users/mpagadal/Data2/ldsc/1000G_plink/EUR/"+x,header=None,delimiter="\t")
#     compiled_bim=compiled_bim.append(bim)
    
# twas_filt=twas[twas["SNP"].isin(compiled_bim[1].tolist())]
# twas_filt.to_csv("/cellar/users/mpagadal/Programs/fusion_twas-master/eur.sumstats.tsv",index=None,sep="\t")

### ../scripts/run_fusion.sh

### Get TWAS results

In [8]:
rsid=pd.read_csv("/cellar/users/mpagadal/resources/rsid/hg19_avsnp147.txt",header=None,delimiter="\t")
rsid["variant"]=rsid[0].astype(str)+":"+rsid[1].astype(str)

/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3049: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [9]:
twas_dir="/cellar/users/mpagadal/Programs/fusion_twas-master/"

files=[x for x in os.listdir(twas_dir) if x.endswith(".dat")]
files=[x for x in files if "top" not in x]

compiled_twas=pd.DataFrame()

for x in files:
    df=pd.read_csv(twas_dir+x,delimiter="\t")
    df["file"]=x
    compiled_twas=compiled_twas.append(df)

compiled_twas["cohort"]=compiled_twas["file"].str.split(".").str[0]
compiled_twas["TWAS.P"]=pd.to_numeric(compiled_twas["TWAS.P"],errors="coerce")
compiled_twas["-log10p"]=-np.log10(compiled_twas["TWAS.P"])

ukbb_twas=compiled_twas[compiled_twas["cohort"]=="ukbb"]
mvp_twas=compiled_twas[compiled_twas["cohort"]=="eur"]

/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/site-packages/ipykernel_launcher.py:15: RuntimeWarning: divide by zero encountered in log10
  from ipykernel import kernelapp as app


In [10]:
len(mvp_twas["BEST.GWAS.ID"].unique())

4840

In [11]:
mvp_twas.to_csv("../data/twas/mvp.testosterone.results.csv",index=None)